# Purchase cleanup

Three card exports, three schemas. The source key is the file name, not a guessed issuer.

| File | Source | Date format | Amount |
| --- | --- | --- | --- |
| Discover-AllAvailable-20261005.csv | `discover` | MM/DD/YYYY | signed Amount |
| Apple Card Transactions Apr 01 2026 - Oct 05 2026.csv | `apple_card` | MM/DD/YYYY | Amount (USD), plus Type |
| 2026-10-05_transaction_download.csv | `txn_download` | YYYY-MM-DD | Debit and Credit columns |

Payment text in the download, such as CAPITAL ONE MOBILE PYMT, shows the issuer. The source key stays `txn_download`.

## What counts as spend

Spend is the sum of `amount` where `include_in_spend` is true. Charges are positive. Payments and refunds are negative and stay out of that sum. Refunds are reported on their own. They are not subtracted from category totals.

- Discover payments are `INTERNET PAYMENT`, `REPRESENTED INTERNET PAYMENT`, and `RETURNED INTERNET PMT`. A returned payment is a reversal: positive amount, still `kind=payment`, not spend. Other negative `Payments and Credits` rows are merchant refunds (Uniqlo, Lego, Mitsuwa).
- Cashback (`Awards and Rebate Credits`) is a statement credit. It is not a payment and not a merchant refund. It is excluded from spend.
- Returned check charges are fees and are included in spend.
- Apple Card types in this file are Purchase, Payment, Installment, and Other. Six installment rows ($149.91 each) are monthly charges. The original financed purchase is not in the export, so those rows are not double-counted. The Apple Store charge typed Other ($143.42) is a separate purchase. The negative transit-offer credit adjustment is a refund.
- In the download, a Credit in `Payment/Credit` is a payment. A Credit in any other category is a refund. A Debit is a purchase. An empty debit or credit cell is blank, not zero.
- Categories are dining, groceries, gas, shopping, travel, transportation, services, medical, other, and payments. This export had no unfamiliar category. Discover `Travel/ Entertainment` is mixed, so transit, airfare, hotel, and car-rental descriptions are split out instead of dumped into one bucket.
- Card numbers are not written. `ENDING IN` suffixes and the card-number column are dropped. One trailing blank line in the download is not a transaction.

No parsed transaction is dropped. Excluded rows stay in the table with `include_in_spend` false.


In [ ]:
import io
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buf.getvalue()))

ROOT = Path.cwd()
if not (ROOT / "src" / "analyze_purchases.py").exists():
    ROOT = Path("/Users/alex/Code/Purchases")
sys.path.insert(0, str(ROOT / "src"))

from analyze_purchases import load_and_clean, sum_amount

DATA_DIR = Path("/Users/alex/Code/Purchases")
df = load_and_clean(DATA_DIR / "statements")
spend = df[df["include_in_spend"]].copy()
print(f"{len(df)} rows  {df['txn_date'].min()} to {df['txn_date'].max()}")
print(f"spend {sum_amount(spend['amount']):.2f} on {len(spend)} rows")
display(df.groupby(["source", "kind"]).size().unstack(fill_value=0))


In [ ]:
by_year = (
    spend.groupby(spend["txn_date"].str.slice(0, 4))["amount"]
    .apply(sum_amount)
    .rename("spend")
)
display(by_year.to_frame())

fig, ax = plt.subplots(figsize=(6, 4))
by_year.plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.set_title("Spend by year")
ax.set_xlabel("")
ax.set_ylabel("Spend")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
show(fig)


In [ ]:
by_month = (
    spend.groupby(spend["txn_date"].str.slice(0, 7))["amount"]
    .apply(sum_amount)
    .sort_index()
    .rename("spend")
)
display(by_month.to_frame())

top = by_month.sort_values(ascending=False).head(5)
print("Most expensive months")
for month, amount in top.items():
    print(f"{month}  {amount:.2f}")

fig, ax = plt.subplots(figsize=(10, 4))
by_month.plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.set_title("Spend by month")
ax.set_xlabel("")
ax.set_ylabel("Spend")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
show(fig)


In [ ]:
by_category = (
    spend.groupby("category")["amount"]
    .apply(sum_amount)
    .sort_values(ascending=False)
    .rename("spend")
)
display(by_category.to_frame())

fig, ax = plt.subplots(figsize=(8, 4))
by_category.sort_values().plot.barh(ax=ax, color="#4c78a8", legend=False)
ax.set_title("Spend by category")
ax.set_xlabel("Spend")
ax.set_ylabel("")
fig.tight_layout()
show(fig)


In [ ]:
# Edit these. Inclusive. Default is the full export range.
START = "2025-03-17"
END = "2026-10-02"

window = df[(df["txn_date"] >= START) & (df["txn_date"] <= END)].copy()
window_spend = window[window["include_in_spend"]].copy()
print(f"Window {START} to {END}")
print(f"spend {sum_amount(window_spend['amount']) if len(window_spend) else 0:.2f} on {len(window_spend)} rows")

by_window_category = (
    window_spend.groupby("category")["amount"]
    .apply(sum_amount)
    .sort_values(ascending=False)
    .rename("spend")
)
print("Category breakdown")
display(by_window_category.to_frame())

by_window_month = (
    window_spend.groupby(window_spend["txn_date"].str.slice(0, 7))["amount"]
    .apply(sum_amount)
    .sort_values(ascending=False)
)
print("Most expensive months in window")
for month, amount in by_window_month.head(5).items():
    print(f"{month}  {amount:.2f}")


In [ ]:
by_source = (
    spend.groupby("source")["amount"]
    .apply(sum_amount)
    .reindex(["discover", "apple_card", "txn_download"])
    .rename("spend")
)
display(by_source.to_frame())

fig, ax = plt.subplots(figsize=(6, 4))
by_source.plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.set_title("Spend by source")
ax.set_xlabel("")
ax.set_ylabel("Spend")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
show(fig)


## Category share by year

Share of spend in each category, within that year. 2025 starts in March and 2026 ends in October, so the dollars are not full-year totals.

In [ ]:
year = spend["txn_date"].str.slice(0, 4)
for name, part in spend.groupby(year):
    print(f"{name}  {part['txn_date'].min()} to {part['txn_date'].max()}  {sum_amount(part['amount']):.2f}")

dollars = (
    spend.groupby([year, "category"])["amount"]
    .apply(sum_amount)
    .unstack("category")
    .fillna(0)
    .sort_index()
)
order = dollars.sum().sort_values(ascending=False).index.tolist()
dollars = dollars[order]
share = dollars.div(dollars.sum(axis=1), axis=0) * 100

shown = dollars.T
shown.columns = [f"{y} $" for y in shown.columns]
for y in dollars.index:
    shown[f"{y} %"] = share.loc[y].round(1)
shown = shown[[col for y in dollars.index for col in (f"{y} $", f"{y} %")]]
display(shown)

colors = {
    "dining": "#4c78a8",
    "shopping": "#f58518",
    "travel": "#e45756",
    "services": "#72b7b2",
    "other": "#54a24b",
    "gas": "#eeca3b",
    "transportation": "#b279a2",
    "groceries": "#ff9da6",
    "medical": "#9d755d",
}
fig, ax = plt.subplots(figsize=(7, 4))
share.plot.bar(stacked=True, ax=ax, color=[colors[c] for c in order], width=0.55, legend=False)
ax.set_ylim(0, 100)
ax.set_title("Category share of spend by year")
ax.set_xlabel("")
ax.set_ylabel("Share of spend (%)")
ax.tick_params(axis="x", rotation=0)
ax.legend(order, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.tight_layout()
show(fig)


## Biggest individual expenses

In [ ]:
biggest = (
    spend.sort_values(["amount", "txn_date"], ascending=[False, True])
    .head(15)
    .loc[:, ["txn_date", "description", "category", "source", "amount"]]
    .reset_index(drop=True)
)
display(biggest)

plot = biggest.iloc[::-1]
labels = plot["txn_date"] + "  " + plot["description"].str.slice(0, 42)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(labels, plot["amount"], color="#4c78a8")
ax.set_title("Biggest individual expenses")
ax.set_xlabel("Amount")
fig.tight_layout()
show(fig)


## Most frequent expenses

Counted on the cleaned description. The same merchant with a different store number or address stays a separate row.

In [ ]:
frequent = (
    spend.groupby("description", sort=False)
    .agg(
        count=("amount", "size"),
        spend=("amount", sum_amount),
        category=("category", "first"),
    )
    .sort_values(["count", "spend"], ascending=[False, False])
    .head(15)
)
display(frequent)

plot = frequent.iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot.index.str.slice(0, 42), plot["count"], color="#4c78a8")
ax.set_title("Most frequent expenses")
ax.set_xlabel("Purchases")
fig.tight_layout()
show(fig)


## 2026 purchases by weekday

In [ ]:
import pandas as pd

weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
s26 = spend[spend["txn_date"].str.startswith("2026")].copy()
s26["weekday"] = pd.to_datetime(s26["txn_date"]).dt.day_name()
by_weekday = pd.DataFrame({
    "purchases": s26.groupby("weekday").size(),
    "spend": s26.groupby("weekday")["amount"].apply(sum_amount),
}).reindex(weekdays)
display(by_weekday)

fig, ax = plt.subplots(figsize=(8, 4))
by_weekday["purchases"].plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.set_title("2026 purchases by weekday")
ax.set_xlabel("")
ax.set_ylabel("Purchases")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
show(fig)


## 2026 payments, January–October

`kind=payment` only. Amounts are shown as money paid (sign flipped). A returned payment would reduce that month.

In [ ]:
import pandas as pd

months = [f"2026-{m:02d}" for m in range(1, 11)]
labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct"]
pay = df[(df["kind"] == "payment") & df["txn_date"].between("2026-01-01", "2026-10-31")].copy()
paid = (
    pay.groupby(pay["txn_date"].str.slice(0, 7))["amount"]
    .apply(lambda s: -sum_amount(s))
    .reindex(months, fill_value=0.0)
)
paid.index = labels
display(paid.rename("paid").to_frame())
print(f"paid {sum_amount(paid):.2f} on {len(pay)} rows")

events = pay.sort_values(["txn_date", "source"]).loc[:, ["txn_date", "source", "description", "amount"]].copy()
events["paid"] = -events["amount"]
display(events.drop(columns="amount").reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8, 4))
paid.plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.set_title("2026 payments, January–October")
ax.set_xlabel("")
ax.set_ylabel("Paid")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
show(fig)


## Dining per month

Average of monthly dining totals over the past 12 complete months, October 2025 through September 2026. October 2026 is only two days of export and has no dining, so it is left out.

In [ ]:
dining = spend[spend["category"] == "dining"].copy()
by_dining_month = (
    dining.groupby(dining["txn_date"].str.slice(0, 7))["amount"]
    .apply(sum_amount)
    .sort_index()
    .rename("dining")
)
past = by_dining_month.loc["2025-10":"2026-09"]
average = sum_amount(past) / len(past)
display(past.to_frame())
print(f"average {average:.2f} over {len(past)} months  {past.index[0]} to {past.index[-1]}")

fig, ax = plt.subplots(figsize=(10, 4))
past.plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.axhline(average, color="#e45756", linewidth=1)
ax.set_title(f"Dining, past 12 months  (average {average:.0f})")
ax.set_xlabel("")
ax.set_ylabel("Dining")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
show(fig)


## Dining per month, weekdays only

Same window, October 2025 through September 2026. Saturday and Sunday are left out.

In [ ]:
import pandas as pd

dining = spend[spend["category"] == "dining"].copy()
dining = dining[dining["txn_date"].between("2025-10-01", "2026-09-30")]
dow = pd.to_datetime(dining["txn_date"]).dt.dayofweek
weekday = dining[dow < 5]
by_weekday_month = (
    weekday.groupby(weekday["txn_date"].str.slice(0, 7))["amount"]
    .apply(sum_amount)
    .sort_index()
    .rename("dining")
)
average = sum_amount(by_weekday_month) / len(by_weekday_month)
display(by_weekday_month.to_frame())
print(f"average {average:.2f} over {len(by_weekday_month)} months  {by_weekday_month.index[0]} to {by_weekday_month.index[-1]}")

fig, ax = plt.subplots(figsize=(10, 4))
by_weekday_month.plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.axhline(average, color="#e45756", linewidth=1)
ax.set_title(f"Weekday dining, past 12 months  (average {average:.0f})")
ax.set_xlabel("")
ax.set_ylabel("Dining")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
show(fig)


## Weekend spend per month

All spend, not just dining. October 2025 through September 2026. Saturday and Sunday only.

In [ ]:
import pandas as pd

window = spend[spend["txn_date"].between("2025-10-01", "2026-09-30")].copy()
dow = pd.to_datetime(window["txn_date"]).dt.dayofweek
weekend = window[dow >= 5]
by_weekend_month = (
    weekend.groupby(weekend["txn_date"].str.slice(0, 7))["amount"]
    .apply(sum_amount)
    .sort_index()
    .rename("spend")
)
average = sum_amount(by_weekend_month) / len(by_weekend_month)
display(by_weekend_month.to_frame())
print(f"average {average:.2f} over {len(by_weekend_month)} months  {by_weekend_month.index[0]} to {by_weekend_month.index[-1]}")

fig, ax = plt.subplots(figsize=(10, 4))
by_weekend_month.plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.axhline(average, color="#e45756", linewidth=1)
ax.set_title(f"Weekend spend, past 12 months  (average {average:.0f})")
ax.set_xlabel("")
ax.set_ylabel("Spend")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
show(fig)


## Past 12 months: weekends vs weekdays

October 2025 through September 2026, excluding 2026-07-01 through 2026-08-15. Saturday and Sunday count as weekend. Monthly averages divide by 10.5 months.

In [ ]:
import pandas as pd

past = spend[spend["txn_date"].between("2025-10-01", "2026-09-30")].copy()
past = past[~past["txn_date"].between("2026-07-01", "2026-08-15")].copy()
past["daytype"] = pd.to_datetime(past["txn_date"]).dt.dayofweek.apply(
    lambda d: "weekend" if d >= 5 else "weekday"
)
by_daytype = pd.DataFrame({
    "spend": past.groupby("daytype")["amount"].apply(sum_amount),
    "purchases": past.groupby("daytype").size(),
}).reindex(["weekday", "weekend"])
by_daytype["average"] = (by_daytype["spend"] / 10.5).round(2)
by_daytype["share"] = (by_daytype["spend"] / by_daytype["spend"].sum() * 100).round(1)
display(by_daytype)

fig, ax = plt.subplots(figsize=(6, 4))
by_daytype["spend"].plot.bar(ax=ax, color="#4c78a8", legend=False)
ax.set_title("Past 12 months: weekdays vs weekends")
ax.set_xlabel("")
ax.set_ylabel("Spend")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
show(fig)


## Past 12 months: category on weekdays and weekends

Same window and same exclusion. Each column is spend inside that day type. Averages divide by 10.5 months.

In [ ]:
import pandas as pd

past = spend[spend["txn_date"].between("2025-10-01", "2026-09-30")].copy()
past = past[~past["txn_date"].between("2026-07-01", "2026-08-15")].copy()
past["daytype"] = pd.to_datetime(past["txn_date"]).dt.dayofweek.apply(
    lambda d: "weekend" if d >= 5 else "weekday"
)
cat_day = (
    past.groupby(["category", "daytype"])["amount"]
    .apply(sum_amount)
    .unstack("daytype")
    .fillna(0)
)
cat_day = cat_day.reindex(cat_day.sum(axis=1).sort_values(ascending=False).index)
cat_day["weekday_average"] = (cat_day["weekday"] / 10.5).round(2)
cat_day["weekend_average"] = (cat_day["weekend"] / 10.5).round(2)
display(cat_day)

fig, ax = plt.subplots(figsize=(10, 5))
cat_day[["weekday", "weekend"]].plot.bar(ax=ax, color=["#4c78a8", "#f58518"])
ax.set_title("Past 12 months: category by weekdays and weekends")
ax.set_xlabel("")
ax.set_ylabel("Spend")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False)
fig.tight_layout()
show(fig)


## Past 12 months: shopping

Shopping per month, plus shopping split into weekdays and weekends. Same exclusion: no July 2026, August is 08-16 onward only. Averages divide by 10.5 months.

In [ ]:
import pandas as pd

past = spend[spend["txn_date"].between("2025-10-01", "2026-09-30")].copy()
past = past[~past["txn_date"].between("2026-07-01", "2026-08-15")].copy()
past["daytype"] = pd.to_datetime(past["txn_date"]).dt.dayofweek.apply(
    lambda d: "weekend" if d >= 5 else "weekday"
)
shopping = past[past["category"] == "shopping"].copy()
by_shopping_month = (
    shopping.groupby(shopping["txn_date"].str.slice(0, 7))["amount"]
    .apply(sum_amount)
    .sort_index()
    .rename("shopping")
)
average = sum_amount(by_shopping_month) / 10.5
display(by_shopping_month.to_frame())
print(f"average {average:.2f} per month over 10.5 months")

by_shopping_daytype = pd.DataFrame({
    "spend": shopping.groupby("daytype")["amount"].apply(sum_amount),
    "purchases": shopping.groupby("daytype").size(),
}).reindex(["weekday", "weekend"])
by_shopping_daytype["average"] = (by_shopping_daytype["spend"] / 10.5).round(2)
display(by_shopping_daytype)

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
by_shopping_month.plot.bar(ax=axes[0], color="#4c78a8", legend=False)
axes[0].axhline(average, color="#e45756", linewidth=1)
axes[0].set_title(f"Shopping by month  (average {average:.0f})")
axes[0].set_xlabel("")
axes[0].set_ylabel("Shopping")
axes[0].tick_params(axis="x", rotation=45)
by_shopping_daytype["spend"].plot.bar(ax=axes[1], color="#4c78a8", legend=False)
axes[1].set_title("Shopping: weekdays vs weekends")
axes[1].set_xlabel("")
axes[1].set_ylabel("Spend")
axes[1].tick_params(axis="x", rotation=0)
fig.tight_layout()
show(fig)


## Unified merchants

`unified_transactions.csv` is every row with a `merchant_key`. Same merchant at different locations shares one key. The table below is spend only, top 30 by total.

In [ ]:
import pandas as pd

unified = pd.read_csv(DATA_DIR / "data" / "unified_transactions.csv")
print(f"{len(unified)} rows across {unified['merchant_key'].nunique()} merchants")

top_merchants = (
    unified[unified["include_in_spend"]]
    .groupby("merchant_key")
    .agg(
        merchant=("merchant_display", "first"),
        purchases=("amount", "size"),
        spend=("amount", sum_amount),
        example=("description", "first"),
    )
    .sort_values("purchases", ascending=False)
)
display(top_merchants.head(50))


# Re-run

From the project root, with the local venv:

```bash
python3 -m venv .venv
.venv/bin/pip install -r requirements.txt
.venv/bin/python src/analyze_purchases.py
.venv/bin/python -m jupyter notebook notebooks/purchases.ipynb
```

The script reads `statements/` and writes `data/cleaned_transactions.csv` and `data/summary.json`. The notebook imports `load_and_clean` from `src/analyze_purchases.py` and does not parse the CSVs itself.

To change the window, edit `START` and `END` in the period cell and run that cell again. Dates are `YYYY-MM-DD` and the filter is inclusive.
